[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/19_imbalanced_learning/01_why_accuracy_lies.ipynb)

# Bài 01 — Dữ liệu mất cân bằng: vì sao accuracy nói dối

**🎯 Mục tiêu**
- Nhận diện bài toán mất cân bằng (gian lận, bệnh hiếm, churn, lỗi sản xuất) và vì sao nó khó.
- Accuracy paradox; metric phù hợp: precision/recall, PR-AUC, recall@k, chi phí.
- Hiểu đúng vấn đề: thường **không** phải tỉ lệ lớp, mà là **số lượng mẫu thiểu số** quá ít + ngưỡng 0.5 không phù hợp.
- Ảnh hưởng của tỉ lệ lớp lên xác suất dự đoán (prior).

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/19_imbalanced_learning

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import metrics as skm
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_course.datasets import make_fraud_transactions
from ml_course.visualization import set_style

set_style()
fraud = make_fraud_transactions(n=60000, fraud_rate=0.01, seed=11)
features = ["amount_vnd", "hour", "distance_from_home_km", "is_foreign", "n_tx_last_24h", "card_age_days"]
X = fraud[features].astype(float).assign(amount_vnd=lambda d: np.log(d.amount_vnd))
X = X.join(pd.get_dummies(fraud.merchant_category, prefix="cat", dtype=float))
y = fraud["is_fraud"].to_numpy()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
print(f"{len(y):,} giao dịch | gian lận: {y.sum()} ({y.mean():.2%}) | trong test: {y_te.sum()}")

## 1. Accuracy paradox

In [ ]:
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)).fit(X_tr, y_tr)
p = model.predict_proba(X_te)[:, 1]
for name, pred in [("luôn đoán 'bình thường'", np.zeros_like(y_te)), ("Logistic Regression, ngưỡng 0.5", (p >= 0.5).astype(int))]:
    print(f"{name:32s} accuracy = {skm.accuracy_score(y_te, pred):.4f} | recall gian lận = {skm.recall_score(y_te, pred):.3f} | "
          f"precision = {skm.precision_score(y_te, pred, zero_division=0):.3f}")

Model "thông minh" có accuracy gần như bằng model vô dụng, và bắt được rất ít gian lận ở ngưỡng 0.5. Nhưng model **không vô dụng** — vấn đề nằm ở ngưỡng và metric:

In [ ]:
print(f"ROC-AUC = {skm.roc_auc_score(y_te, p):.3f} | PR-AUC = {skm.average_precision_score(y_te, p):.3f} (ngẫu nhiên = {y_te.mean():.3f})")
order = np.argsort(-p)
for k in [50, 200, 1000]:
    tp = y_te[order[:k]].sum()
    print(f"top {k:4d} giao dịch rủi ro nhất: {tp} gian lận → precision@k = {tp / k:.2f}, recall@k = {tp / y_te.sum():.2f}")

Xếp hạng tốt (PR-AUC gấp nhiều lần ngẫu nhiên, top-k chứa nhiều gian lận) — chỉ là xác suất dự đoán hiếm khi vượt 0.5 vì gian lận vốn hiếm.

## 2. Vì sao xác suất "thấp"? — prior

Model calibrated dự đoán $P(\text{gian lận} \mid \mathbf x)$. Theo Bayes, xác suất này **tỉ lệ với prior** $P(\text{gian lận}) = 1\%$: một giao dịch đáng ngờ gấp 30 lần trung bình cũng chỉ có $P \approx 0.3$. Ngưỡng 0.5 ngầm giả định FP và FN đắt như nhau — với gian lận, bỏ sót đắt hơn nhiều.

In [ ]:
plt.hist(p[y_te == 0], bins=60, alpha=0.6, density=True, label="bình thường")
plt.hist(p[y_te == 1], bins=60, alpha=0.6, density=True, label="gian lận")
plt.axvline(0.5, color="k", ls="--", label="ngưỡng 0.5")
plt.yscale("log")
plt.xlabel("P(gian lận) dự đoán")
plt.legend()
plt.title("Gian lận có xác suất cao hơn hẳn — nhưng phần lớn vẫn < 0.5")
plt.show()

## 3. Vấn đề thật: quá ít mẫu thiểu số

Hai thí nghiệm với Logistic Regression (mỗi cấu hình lặp 5 lần với mẫu con khác nhau, lấy trung bình PR-AUC trên cùng tập test):
- **(a)** giữ tỉ lệ ~1%, thay đổi **tổng số mẫu** (tức số gian lận);
- **(b)** giữ **mọi** gian lận, bỏ bớt lớp đa số để tỉ lệ "cân bằng" hơn.

In [ ]:
def train_eval(idx):
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)).fit(X_tr.iloc[idx], y_tr[idx])
    return skm.average_precision_score(y_te, m.predict_proba(X_te)[:, 1])


pos_idx, neg_idx = np.flatnonzero(y_tr == 1), np.flatnonzero(y_tr == 0)
rows = []
for frac in [0.1, 0.3, 1.0]:
    scores = [train_eval(np.r_[np.random.default_rng(s).choice(pos_idx, int(frac * len(pos_idx)), replace=False),
                               np.random.default_rng(s + 100).choice(neg_idx, int(frac * len(neg_idx)), replace=False)]) for s in range(5)]
    rows.append({"thí nghiệm": "(a) cùng tỉ lệ ~1%", "số gian lận train": int(frac * len(pos_idx)),
                 "số bình thường train": int(frac * len(neg_idx)), "PR-AUC test (TB 5 lần)": np.mean(scores)})
for n_neg in [len(neg_idx), 20000, 3000]:
    scores = [train_eval(np.r_[pos_idx, np.random.default_rng(s).choice(neg_idx, n_neg, replace=False)]) for s in range(5)]
    rows.append({"thí nghiệm": "(b) giữ mọi gian lận", "số gian lận train": len(pos_idx), "số bình thường train": n_neg,
                 "PR-AUC test (TB 5 lần)": np.mean(scores)})
pd.DataFrame(rows).round(4)

- **(a)** Với chỉ ~40 mẫu gian lận, model tệ đi rõ rệt; từ vài trăm mẫu trở lên, model tuyến tính đơn giản này đã bão hoà (model phức tạp hơn cần nhiều mẫu thiểu số hơn nữa). Khi model còn "đói" dữ liệu thiểu số, **thu thập/gán nhãn thêm** thường là cách hiệu quả nhất.
- **(b)** Bỏ bớt lớp đa số để "cân bằng" (41 583 → 3 000) **không** làm model tốt hơn — chỉ tiết kiệm thời gian train, và làm sai lệch xác suất (bài 02).

## 4. Checklist cho bài toán mất cân bằng

1. **Metric**: PR-AUC, recall/precision tại ngưỡng vận hành, precision@k, chi phí — không dùng accuracy.
2. **Chia dữ liệu**: stratified; đủ mẫu thiểu số trong validation/test (đánh giá rất nhiễu khi chỉ có vài chục positive — báo cáo CI).
3. **Ngưỡng**: chọn theo chi phí/năng lực vận hành (chương 14).
4. **Model**: thử class weight; resampling chỉ khi cần (bài 02–03); model mạnh (boosting) thường xử lý tốt.
5. **Quá mất cân bằng** (< 0.1%, ít nhãn): cân nhắc anomaly detection (chương 22) hoặc học bán giám sát.

## 🧠 Kiểm tra nhanh

1. Dữ liệu có 0.5% positive. Accuracy của model "luôn đoán negative"?
2. Vì sao model calibrated hiếm khi cho xác suất > 0.5 với lớp rất hiếm?
3. Giữa (a) gấp đôi số mẫu thiểu số và (b) cân bằng tỉ lệ bằng cách bỏ lớp đa số — cái nào thường giúp nhiều hơn?
4. Kể 3 metric phù hợp với dữ liệu mất cân bằng.

## 📌 Tóm tắt

- Accuracy vô nghĩa khi mất cân bằng; dùng PR-AUC, precision/recall tại ngưỡng, @k, chi phí.
- Xác suất thấp là do prior — vấn đề là ngưỡng, không phải model.
- Số mẫu thiểu số quan trọng hơn tỉ lệ lớp.

## 📚 Tài liệu tham khảo

- He & Garcia (2009) — *Learning from Imbalanced Data*
- Saito & Rehmsmeier (2015) — *The Precision-Recall Plot Is More Informative than the ROC Plot...*
- van den Goorbergh et al. (2022) — *The harm of class imbalance corrections for risk prediction models*